In [3]:
import numpy as np
import matplotlib.pyplot as plt

# Add files up one level to the path
import sys
# sys.path.append("..")

import jlinops

from matplotlib.backends.backend_pdf import PdfPages


from scipy.linalg import qr, qr_insert
from scipy.integrate import odeint
import scipy.optimize as op
import scipy.sparse as sps

from fastprogress import progress_bar


# Try to import matlab.engine
try:
    import matlab.engine
except:
    pass # assume we don't need it
    # raise Exception("Error with matlab.engine, make sure it is installed correctly.")




In [12]:
from trips.utilities.weights import smoothed_holder_weights

from trips.utilities.reg_param.discrepancy_principle import discrepancy_principle
from trips.utilities.reg_param.gcv import generalized_crossvalidation
from regparam_selection import mgcv, dp_cubic, matlab_gcv

In [16]:
class GKSSolver:
    """Implements a generic GKS solver for problems of the form:

    If noise_var is provided, then the problem solved is 
        argmin_x  (1/noise_var)*|| A x - b ||_2^2 + \lambda || L x ||_q^q.

    """

    def __init__(self, A, L, b, noise_var=None):

        # Bind
        self._A = A
        self.L = L
        self._b = b
        self.noise_var = noise_var
        self.n = A.shape[1]
        self.m = A.shape[0]
        self.k = L.shape[0]

        # Whiten?
        if self.noise_var is not None:
            self.noise_stdev = np.sqrt(self.noise_var)
            self.A = (1.0/self.noise_stdev)*self._A
            self.b = (1.0/self.noise_stdev)*self._b
        else:
            self.noise_stdev = None
            self.A = self._A
            self.b = self._b


    def build_initial_basis(self, dim):
        """
        Builds the initial basis.
        """

        Atb = self.A.T @ self.b

        V = np.zeros((self.n, dim))
        V[:,0] = Atb
        tmp = Atb.copy()
        for j in range(1,dim):
            tmp = self.A.T @ (self.A @ tmp)
            V[:,j] = tmp.copy()

        # V = np.random.normal(size=(self.n, 5))

        # Orthogonalize
        qr_fac = np.linalg.qr(V)
        V = qr_fac.Q

        return V
    

    def select_regparam(self, Q_A, R_A, Q_L, R_L, regparam, eng=None, **regparam_kwargs):
        """Selects regularization parameter for the projected problem.
        """

        # Select new regularization parameter
        if regparam == "mgcv": # Jonathan's MGCV method
            proposed_lambdah = mgcv(Q_A, R_A, R_L, self.b, **regparam_kwargs)
        elif regparam == "dp": # Trips-Py dp
            proposed_lambdah = discrepancy_principle(Q_A, R_A, R_L, self.b, delta=np.sqrt(self.m), explicitProj=True, **regparam_kwargs)
        elif regparam == 'gcv': # Trips-Py gcv
            proposed_lambdah = generalized_crossvalidation(Q_A, R_A, R_L, self.b, **regparam_kwargs)
        elif regparam == "dp2": # Jonathan's 2nd-order DP code. Requires 2 LS solves per iteration.
            shift = np.linalg.norm( self.b -  ( Q_A @ (Q_A.T @ self.b) ), ord=2 )**2 # needed for calculation w.r.t. full-scale objective
            # Hacky try-except block since code is not completely debugged for small \beta!
            try:
                data = dp_quadratic(R_A, R_L, Q_A.T @ self.b, noise_var=1.0, shift=shift, m=len(self.b), beta0=self.beta0, **regparam_kwargs)
            except:
                # If it didn't work with initializing beta0, try again without initializing it (defaults to beta0 very small). 
                data = dp_quadratic(R_A, R_L, Q_A.T @ self.b, noise_var=1.0, shift=shift, m=len(self.b), beta0=None, **regparam_kwargs)
                #pass
            self.beta0 = 1.0/proposed_lambdah
            proposed_lambdah = data["lambda"]
        elif regparam == "dp3": # Jonathan's 3rd-order DP code. Requires 3 LS solves per iteration.
            shift = np.linalg.norm( self.b -  ( Q_A @ (Q_A.T @ self.b) ), ord=2 )**2 # needed for calculation w.r.t. full-scale objective
            # Hacky try-except block since code is not completely debugged for small \beta!
            try:
                data = dp_cubic(R_A, R_L, Q_A.T @ self.b, noise_var=1.0, shift=shift, m=len(self.b), beta0=self.beta0, **regparam_kwargs)
            except:
                # If it didn't work with initializing beta0, try again without initializing it (defaults to beta0 very small). 
                data = dp_cubic(R_A, R_L, Q_A.T @ self.b, noise_var=1.0, shift=shift, m=len(self.b), beta0=None, **regparam_kwargs)
                #pass
            self.beta0 = 1.0/proposed_lambdah
            proposed_lambdah = data["lambda"]
        elif regparam == "matlab_gcv": # MATLAB GCV code
            proposed_lambdah = matlab_gcv( R_A, R_L, Q_A.T @ self.b, eng=eng, **regparam_kwargs)
        elif not isinstance(regparam, str): # hope that regparam is a scalar
            proposed_lambdah = regparam
        else:
            raise NotImplementedError

        return proposed_lambdah


    def solve(self, n_iter=25, init_dim=5, regparam="dp", x0=None, x_true=None, weight_params=None, regparam_kwargs={}, **kwargs):
        """Driver for GKS solver.
        """

        # Tracking
        tot_matvec_A = 0
        tot_matvec_L = 0 
        lambda_history = []
        residual_history = []
        rre_history = []
        cond_ests = []

        # Initialization
        if x0 is None:
            x = self.A.T @ self.b
            self.tot_matvec_A += 1
        else:
            x = x0

        if x_true is not None:
            x_true_norm = np.linalg.norm(x_true)
            rre_history.append( np.linalg.norm(x - x_true)/x_true_norm )

        # Build initial basis and QR factor of AV
        V = self.build_initial_basis(init_dim)
        self.tot_matvec_A += (2*(init_dim - 1)) + 1 
        AV = self.A @ V 
        tot_matvec_A += init_dim
        Q_A, R_A = qr(AV, mode='economic') 

        # Get initial LV
        LV = self.L @ V
        tot_matvec_L += init_dim

        # For cubic dp
        self.beta0 = None

        # Start matlab engine if regparam == matlab_gcv
        _eng = None
        if regparam == "matlab_gcv":
            # If this throws an error, it is because matlab engine is not set up correctly
            _eng = matlab.engine.start_matlab()

        for ii in progress_bar(range(n_iter)):

            # Update weights and get new QR factorization of W L V
            Lx = self.L @ x
            tot_matvec_L += 1
            w = self.update_weights(x, weight_params)
            W = sps.diags(w)
            WLV = W @ LV
            Q_WL, R_WL = qr(WLV, mode='economic') 

            # Select new regularization parameter
            proposed_lambdah = self.select_regparam(Q_A, R_A, Q_WL, R_WL, regparam=regparam, eng=_eng, **regparam_kwargs)
            
            # Screen the proposed lambdah somehow? Just accept for now...
            lambdah = proposed_lambdah
            lambda_history.append(lambdah)
            try:
                beta0 = 1.0/lambdah
            except:
                beta0 = 1e10

            # Obtain new solution
            y, _ , _, svals = np.linalg.lstsq(np.concatenate((R_A, np.sqrt(lambdah) * R_WL)), np.concatenate((Q_A.T @ self.b, np.zeros((R_WL.shape[0],)))), rcond=None)
            cond_est = np.amax(svals)/np.amin(svals)
            cond_ests.append(cond_est)
            x = V @ y

            if x_true is not None: rre_history.append( np.linalg.norm(x - x_true)/x_true_norm )

            if ii >= R_WL.shape[0]:
                break
            
            # Get new residual
            v = AV @ y
            v = v - self.b
            u = LV @ y
            ra = AV @ y - self.b
            ra = self.A.T @ ra
            rb = W @ (LV @ y)
            rb = self.L.T @ rb
            r = ra + lambdah * rb
            r = r - V @ (V.T @ r) # re-orthogonalize
            vn = r / np.linalg.norm(r)
            V = np.column_stack((V, vn))
            Avn = self.A @ vn
            AV = np.column_stack((AV, Avn))
            Lvn = vn
            Lvn = self.L * vn
            LV = np.column_stack((LV, Lvn))
            tot_matvec_A += 2
            tot_matvec_L += 2
            residual_history.append(np.linalg.norm(r))

            # Update QR factor for A
            #Q_A, R_A = qr_insert( Q_A, R_A, u = Avn, k=-1, which="col" )
            Q_A, R_A = qr(AV, mode='economic') 


        # Package output
        data = {
            "x": x,
            "w": w,
            "tot_matvec_A": tot_matvec_A,
            "tot_matvec_L": tot_matvec_L,
            "residual_history": np.asarray(residual_history),
            "lambda_history": np.asarray(lambda_history),
            "V": V,
            "cond_ests": np.asarray(cond_ests),
            #"dp_cubic_n_iters": np.asarray(dp_cubic_n_iters),
        }

        if x_true is not None:
            data["rre_history"] = np.asarray(rre_history)

        # Close engine if need to
        if regparam == "matlab_gcv": _eng.close()

        return data

    
    def solve_with_recycling(self):
        """Driver for GKS solver with recycling.
        """
        raise NotImplementedError


    def solve_with_priorconditioning(self):
        """Driver for GKS solver with priorconditioning.
        """
    
        raise NotImplementedError
    

    def update_weights(self, x, weight_params):

        raise NotImplementedError

In [17]:
class MMGKS(GKSSolver):

    def __init__(self, A, L, b, noise_var=None):

        super().__init__(A, L, b, noise_var=noise_var)


    def update_weights(self, Lx, weight_params):
        
        w = smoothed_holder_weights( Lx, epsilon=weight_params["epsilon"], p=weight_params["q"])

        return w

In [18]:
class IASGKS(GKSSolver):

    def __init__(self, A, L, b, noise_var=None):

        super().__init__(A, L, b, noise_var=noise_var)


    def update_weights(self, Lx, weight_params):
        
        w = smoothed_holder_weights( Lx, epsilon=weight_params["epsilon"], p=weight_params["q"])

        return w

In [19]:
class VanillaGKS(GKSSolver):

    def __init__(self, A, L, b, noise_var=None):

        super().__init__(A, L, b, noise_var=noise_var)


    def update_weights(self, Lx, weight_params):

        return np.ones_like(Lx)
    